In [ ]:
# !pip install google-cloud-bigquery google-cloud-storage

In [ ]:
import os
# os.environ['GOOGLE_CLOUD_PROJECT'] = 'rs-nprd-dlk-agspc-roy-5b05'

In [ ]:
# service_account_email = 'sa-nprd-dt-iam-royspc-dflow2@rs-nprd-dlk-agspc-roy-5b05.iam.gserviceaccount.com'

In [1]:
from google.colab import auth
auth.authenticate_user()

In [2]:
####CONSULTA DE ARCHIVOS #####
from google.cloud import storage
import zipfile
import io
import re
import os

# Configuración
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
PREFIX = "data_entries/Trama_Sustento/"  # Carpeta en el bucket
# PATTERN = re.compile(r"RED\d{6}_\d{4}\.txt$", re.IGNORECASE)  # Expresión regular para el patrón
PATTERN = re.compile(r".*\.txt$", re.IGNORECASE)

def list_txt_files_with_pattern(bucket_name, prefix, pattern):
    """Lista los nombres de archivos TXT dentro de archivos ZIP, incluyendo aquellos en subcarpetas."""
    storage_client = storage.Client()
    bucket = storage_client.bucket(bucket_name)
    blobs = bucket.list_blobs(prefix=prefix)

    # Iterar sobre todos los archivos en la ruta especificada
    for blob in blobs:
        if blob.name.endswith('.zip'):  # Filtrar solo archivos ZIP
            print(f"Procesando archivo ZIP: {blob.name}")

            # Descargar el ZIP en memoria
            zip_bytes = blob.download_as_bytes()

            # Abrir el ZIP en memoria
            with zipfile.ZipFile(io.BytesIO(zip_bytes), 'r') as z:
                for file_path in z.namelist():  # Recorremos todos los archivos dentro del ZIP
                    file_name = os.path.basename(file_path)  # Extraer solo el nombre del archivo ignorando carpetas
                    if pattern.match(file_name):  # Aplicar la expresión regular al nombre limpio
                        print(f"Archivo encontrado: {file_name} en {blob.name} (Ruta interna: {file_path})")

# Ejecutar la función
list_txt_files_with_pattern(BUCKET_NAME, PREFIX, PATTERN)


Procesando archivo ZIP: data_entries/Trama_Sustento/SUST0625.zip
Archivo encontrado: SUST0625.TXT en data_entries/Trama_Sustento/SUST0625.zip (Ruta interna: SUST0625.TXT)
Procesando archivo ZIP: data_entries/Trama_Sustento/SUST0725.zip
Archivo encontrado: SUST0725.txt en data_entries/Trama_Sustento/SUST0725.zip (Ruta interna: SUST0725.txt)


# CON TABLA DE CONTROL

In [3]:
"""
Código COMPLETO E INTEGRADO con verificación de dataset y tabla de control.
"""

import zipfile
import io
import os
import re
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow as pa
from datetime import datetime
from google.cloud import storage, bigquery
from google.cloud.exceptions import NotFound
from itertools import accumulate


# =====================
# CONFIGURACIÓN
# =====================
PROJECT_ID = "rs-nprd-dlk-agspc-roy-5b05"
BUCKET_NAME = "rs-nprd-dlk-ue4-gcs-ryl-sftp_generics"
PREFIX = "data_entries/Trama_Sustento/"
DATASET_ID = "develop"
TABLE_PRIMA_ID = "TABLA_SUSTENTO_BBVA"
TABLE_CONTROL_ID = "TABLA_CONTROL_SUSTENTO"
GCS_TEMP_PATH = f"gs://{BUCKET_NAME}/temp_parquet_sust/"

# =====================
# CLIENTES DE GCP
# =====================
storage_client = storage.Client(project=PROJECT_ID)
bigquery_client = bigquery.Client(project=PROJECT_ID)

# =====================
# EXPRESIÓN REGULAR
# =====================
PATTERN = re.compile(r"^SUST(\d{2})(\d{2})(?:_\d+)?\.TXT$", re.IGNORECASE)

# =====================
# ESTRUCTURA DEL TXT
# =====================

# Definir los anchos de columna
col_widths = [4, 30, 3, 20, 3, 10, 20, 20, 1, 10, 10, 15, 15, 15, 2, 18, 15, 10, 2, 2, 7, 8, 8, 8, 2, 15, 2, 15, 15]
column_names = ['Oficina','Descripcion_Oficina','Codigo_Subproducto','Descripcion_Producto','Moneda','Nro_Poliza','Nro_Certificado','Nro_Cuenta','Forma_de_Pago','Fecha_de_Cobro','Fecha_de_Liquidacion','Prima','Comision','%_Comision','Indicador_de_Renovacion','Filler','Monto_Asegurado','Tasa_sin_Recargo','Plan_de_credito','Planes_Tipo_Desgravamen','%_recargo','Fec_Inicio_Pago','Fec_Fin_Pago','Fecha_Inicio_de_Vigencia_Poliza','Modalidad','Prima_Rimac','Compañia','Prima_Desempleo','Prima_Desgravamen']

# =====================
# LOG
# =====================
def log(message: str) -> None:
    print(f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')}] {message}")


# =====================
# EXPRESIÓN REGULAR -> FECHA
# =====================

def extract_fecha_recepcion(file_name: str) -> str:
    match = PATTERN.match(file_name)
    if not match:
        return None

    mm, aa = match.groups()

    # Convertir año
    yy = int(aa)
    full_year = 2000 + yy if yy < 50 else 1900 + yy  # Si yy < 50 => 20xx, de lo contrario 19xx

    # Día fijo = 15
    return f"{full_year}-{mm}-15"

# =====================
# TABLA PRINCIPAL
# =====================
def create_table_if_not_exists():
    """
    Crea la tabla principal en BigQuery con particionamiento
    si no existe. Si ya existe, no se modifica su esquema.
    En esta versión, el campo 'Prima' se define como FLOAT.
    """
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_PRIMA_ID}"

    # Definir el esquema:
    # - Para 'Prima' se define FLOAT.
    # - Para el resto de las columnas se utiliza STRING.
    schema = []
    for col in column_names:
        if col in ["Prima", "Prima_Rimac", "Prima_Desempleo", "Prima_Desgravamen", "Monto_Asegurado"]:
            schema.append(bigquery.SchemaField(col, "FLOAT"))
        else:
            schema.append(bigquery.SchemaField(col, "STRING"))
    schema += [
        bigquery.SchemaField("trama_original", "STRING"),
        bigquery.SchemaField("nombre_archivo_trama", "STRING"),
        bigquery.SchemaField("fecha_recepcion", "DATE"),
    ]

    # ==== Imprimir el esquema que se generó ====
    log(">>> Esquema a usar para crear la tabla:")
    # for field in schema:
    #     log(f" - {field.name}: {field.field_type}")

    try:
        bigquery_client.get_table(table_id)
        log(f"✅ La tabla {TABLE_PRIMA_ID} ya existe (no se modifica).")
    except NotFound:
        # Crear la tabla con particionamiento únicamente
        table = bigquery.Table(table_id, schema=schema)

        # Particionamiento por fecha_recepcion
        table.time_partitioning = bigquery.TimePartitioning(field="fecha_recepcion")

        bigquery_client.create_table(table)
        log(f"✅ Tabla {TABLE_PRIMA_ID} creada con particionamiento en BigQuery.")

# =====================
# TABLA DE CONTROL
# =====================
def create_control_table():
    """
    Crea la tabla de control en BigQuery si no existe.
    """
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}"

    schema = [
        bigquery.SchemaField("nombre_archivo", "STRING"),
        bigquery.SchemaField("fecha_carga", "TIMESTAMP"),
        bigquery.SchemaField("estado", "STRING"),
    ]

    try:
        bigquery_client.get_table(table_id)
        log(f"✅ La tabla de control {TABLE_CONTROL_ID} ya existe.")
    except NotFound:
        table = bigquery.Table(table_id, schema=schema)
        bigquery_client.create_table(table)
        log(f"✅ Tabla de control {TABLE_CONTROL_ID} creada.")
        # Pequeño retardo para asegurar la propagación de metadatos
        time.sleep(5)

# =====================
# VALIDACIÓN DE PROCESADO DE ZIP
# =====================
def check_if_zip_processed(file_name: str) -> bool:
    """Verifica si el ZIP ya fue procesado (estado = 'CARGADO') en la tabla de control."""
    query = f"""
        SELECT COUNT(*) AS count
        FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}`
        WHERE nombre_archivo = '{file_name}' AND estado = 'CARGADO'
    """
    df = bigquery_client.query(query).to_dataframe()
    return df['count'][0] > 0

# =====================
# VALIDACIÓN DE PROCESADO DE PARQUET
# =====================
def check_if_parquet_processed(parquet_name: str) -> bool:
    """Verifica si el Parquet ya fue procesado (estado = 'CARGADO_PARQUET') en la tabla de control."""
    query = f"""
        SELECT COUNT(*) AS count
        FROM `{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}`
        WHERE nombre_archivo = '{parquet_name}' AND estado = 'CARGADO_PARQUET'
    """
    df = bigquery_client.query(query).to_dataframe()
    return df['count'][0] > 0

# =====================
# REGISTRO EN TABLA DE CONTROL CON REINTENTOS
# =====================
def insert_control_record(file_name: str, estado: str, max_retries: int = 3, delay: int = 5) -> None:
    """
    Registra el estado del archivo (ZIP o PARQUET) en la tabla de control.
    - 'CARGADO' para ZIP
    - 'CARGADO_PARQUET' para Parquet
    Se reintenta la inserción en caso de error NotFound.
    """
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_CONTROL_ID}"
    rows_to_insert = [{
        "nombre_archivo": file_name,
        "fecha_carga": datetime.utcnow().isoformat(),
        "estado": estado
    }]

    for attempt in range(1, max_retries + 1):
        try:
            errors = bigquery_client.insert_rows_json(table_id, rows_to_insert)
            if errors:
                log(f"❌ Errores al insertar en la tabla de control: {errors}")
            else:
                log(f"✅ Registro insertado en tabla de control para {file_name} (estado: {estado}).")
            break
        except NotFound:
            log(f"❌ Intento {attempt}: La tabla {TABLE_CONTROL_ID} no se encontró. Esperando {delay} segundos...")
            time.sleep(delay)
    else:
        log(f"❌ Error: No se pudo insertar el registro en {TABLE_CONTROL_ID} tras {max_retries} intentos.")

# =====================
# FUNCIÓN DE CONVERSIÓN ESPECÍFICA
# =====================
def convert_prima(series: pd.Series) -> pd.Series:
    """
    Optimiza la conversión para columnas en las que se reemplaza el último carácter
    según un diccionario, por ejemplo: 'Plazo_del_seguro', 'Monto_Asegurado', 'Prima'.
    """
    clave = np.array(list("ABCDEFGHIJKLMNOPQRSTUVWXYZ{0123456789"))
    valor = np.array(list("1234567891234567890000000000123456789"))
    diccionario_reemplazo = dict(zip(clave, valor))

    series = series.fillna("0").astype(str)

    return pd.Series(
        np.where(
            (series == "0") | (series == ""),
            series,
            series.str[:-1] + series.str[-1].map(diccionario_reemplazo).fillna("")
        ),
        index=series.index
    )

# =====================
# PROCESAMIENTO DE TXT
# =====================
def process_txt_file(txt_file, file_name: str) -> pd.DataFrame:
    """
    Procesa un archivo TXT a DataFrame y lo guarda en Parquet en GCS.

    Parámetros:
      - txt_file: Archivo TXT abierto en modo binario.
      - file_name: Nombre del archivo (se utiliza para extraer la fecha y registrar en control).

    Retorna:
      - pd.DataFrame con los datos procesados.
    """
    fecha_recepcion = extract_fecha_recepcion(file_name)
    if not fecha_recepcion:
        log(f"❌ No se pudo extraer la fecha de recepción del archivo: {file_name}")
        return pd.DataFrame()

    # Leer y decodificar el contenido del archivo
    lines = txt_file.read().decode('latin-1').splitlines()

    # Precalcular los índices de corte para cada columna utilizando accumulate
    cumulative_indices = [0] + list(accumulate(col_widths))

    data = []
    for line in lines:
        # Extraer cada campo de acuerdo a los anchos definidos y quitar espacios en blanco
        row = {
            col: line[start:end].strip()
            for col, start, end in zip(column_names, cumulative_indices[:-1], cumulative_indices[1:])
        }
        # Agregar campos adicionales
        row.update({
            "trama_original": line.rstrip("\n"),
            "nombre_archivo_trama": file_name,
            "fecha_recepcion": fecha_recepcion
        })
        data.append(row)

    # Crear DataFrame con las columnas en el orden esperado
    df = pd.DataFrame(data, columns=column_names + ["trama_original", "nombre_archivo_trama", "fecha_recepcion"])

    # Convertir "fecha_recepcion" a tipo fecha
    df["fecha_recepcion"] = pd.to_datetime(df["fecha_recepcion"], format="%Y-%m-%d").dt.date

    # Lista de columnas que requieren conversión a numérico (dividido entre 100)
    columnas_primas = ["Prima", "Prima_Rimac", "Prima_Desempleo", "Prima_Desgravamen", "Monto_Asegurado"]
    for col in columnas_primas:
        if col in df.columns:
            df[col] = pd.to_numeric(convert_prima(df[col]), errors="coerce") / 100.0

    # Guardar el DataFrame en formato Parquet en GCS
    file_parquet = f"{GCS_TEMP_PATH}{file_name}.parquet"
    table = pa.Table.from_pandas(df)
    pq.write_table(table, file_parquet)
    log(f"📤 Guardado {file_parquet} en GCS.")

    return df

# =====================
# CARGA PARQUET A BQ (Control de duplicados)
# =====================
def load_parquet_to_bigquery():
    """
    Carga los archivos Parquet de GCS a la tabla principal en BigQuery,
    evitando duplicados mediante la tabla de control.
    """
    table_id = f"{PROJECT_ID}.{DATASET_ID}.{TABLE_PRIMA_ID}"
    job_config = bigquery.LoadJobConfig(
        source_format=bigquery.SourceFormat.PARQUET,
        write_disposition="WRITE_APPEND"
    )

    bucket = storage_client.bucket(BUCKET_NAME)
    prefix_parquet = GCS_TEMP_PATH.replace(f"gs://{BUCKET_NAME}/", "")
    blobs = list(bucket.list_blobs(prefix=prefix_parquet))

    if not blobs:
        log("⚠️ No se encontraron archivos PARQUET en la ruta.")
        return

    for blob in blobs:
        if blob.name.endswith(".parquet"):
            parquet_name = os.path.basename(blob.name)

            # Verificar si ya se cargó
            if check_if_parquet_processed(parquet_name):
                log(f"🔄 Parquet ya cargado: {parquet_name}. Omitiendo...")
                continue

            # Cargar este Parquet en BQ
            parquet_uri = f"gs://{BUCKET_NAME}/{blob.name}"
            log(f"🚀 Cargando {parquet_name} a BigQuery...")

            load_job = bigquery_client.load_table_from_uri(
                parquet_uri,
                table_id,
                job_config=job_config
            )
            load_job.result()

            log(f"✅ Cargado Parquet {parquet_name} en BigQuery.")
            insert_control_record(parquet_name, "CARGADO_PARQUET")

    log("🚀 Carga de todos los Parquets completada en BigQuery.")

# =====================
# PROCESAR ZIP
# =====================
def process_zip_files():
    """
    Procesa archivos ZIP en GCS, extrayendo los que matchean PATTERN,
    incluso en subcarpetas, y evita reprocesarlos si el Parquet ya existe.
    """
    bucket = storage_client.bucket(BUCKET_NAME)
    blobs = list(bucket.list_blobs(prefix=PREFIX))

    if not blobs:
        log("⚠️ No se encontraron archivos en el bucket.")
        return

    for blob in blobs:
        if blob.name.endswith(".zip"):
            file_name = os.path.basename(blob.name)

            # Chequear si el ZIP ya fue procesado
            if check_if_zip_processed(file_name):
                log(f"🔄 Archivo ZIP ya procesado: {file_name}. Omitiendo...")
                continue

            log(f"\n📦 Procesando ZIP: {file_name}")
            zip_bytes = blob.download_as_bytes()

            with zipfile.ZipFile(io.BytesIO(zip_bytes), "r") as z:
                all_files = z.namelist()
                log(f"Archivos dentro del ZIP: {all_files}")

                # Filtrar archivos .txt que matchean la regex
                archivos_txt = []
                for path_in_zip in all_files:
                    base = os.path.basename(path_in_zip)
                    if PATTERN.match(base):
                        archivos_txt.append(path_in_zip)

                if not archivos_txt:
                    log(f"⚠️ No se encontraron TXT válidos en {file_name}. Omitiendo...")
                    continue

                for txt_path in archivos_txt:
                    base_txt_name = os.path.basename(txt_path)
                    parquet_file = f"{base_txt_name}.parquet"
                    full_parquet_gcs_path = f"{GCS_TEMP_PATH}{parquet_file}"

                    # Revisamos si el Parquet ya existe en el bucket
                    parquet_blob = bucket.blob(full_parquet_gcs_path.replace(f"gs://{BUCKET_NAME}/", ""))
                    if parquet_blob.exists():
                        log(f"🔄 Parquet {parquet_file} ya existe. Omitiendo procesamiento de {base_txt_name}...")
                        continue

                    log(f"→ Procesando archivo: {txt_path} (basename={base_txt_name})")
                    with z.open(txt_path) as txt_file:
                        df = process_txt_file(txt_file, base_txt_name)
                        if not df.empty:
                            # Marcar el ZIP como procesado
                            insert_control_record(file_name, "CARGADO")

# =====================
# MAIN
# =====================
if __name__ == "__main__":
    log("🚀 Iniciando proceso...")

    # 2. Crear tablas si no existen, con particionamiento y clustering
    create_table_if_not_exists()
    create_control_table()

    # 3. Procesar ZIP y extraer a Parquet (evita recrear Parquet si ya existe)
    process_zip_files()

    # 4. Cargar los Parquets a la tabla principal (evitando duplicados)
    load_parquet_to_bigquery()

    log("✅ Proceso finalizado.")


[2025-08-05 20:26:18] 🚀 Iniciando proceso...
[2025-08-05 20:26:18] >>> Esquema a usar para crear la tabla:
[2025-08-05 20:26:19] ✅ La tabla TABLA_SUSTENTO_BBVA ya existe (no se modifica).
[2025-08-05 20:26:19] ✅ La tabla de control TABLA_CONTROL_SUSTENTO ya existe.
[2025-08-05 20:26:21] 🔄 Archivo ZIP ya procesado: SUST0625.zip. Omitiendo...
[2025-08-05 20:26:23] 
📦 Procesando ZIP: SUST0725.zip
[2025-08-05 20:26:23] Archivos dentro del ZIP: ['SUST0725.txt']
[2025-08-05 20:26:23] → Procesando archivo: SUST0725.txt (basename=SUST0725.txt)
[2025-08-05 20:27:38] 📤 Guardado gs://rs-nprd-dlk-ue4-gcs-ryl-sftp_generics/temp_parquet_sust/SUST0725.txt.parquet en GCS.
[2025-08-05 20:27:40] ✅ Registro insertado en tabla de control para SUST0725.zip (estado: CARGADO).
[2025-08-05 20:27:44] 🔄 Parquet ya cargado: SUST0425.TXT.parquet. Omitiendo...
[2025-08-05 20:27:47] 🔄 Parquet ya cargado: SUST0525.TXT.parquet. Omitiendo...
[2025-08-05 20:27:49] 🔄 Parquet ya cargado: SUST0625.TXT.parquet. Omitiendo..